In [1]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash
from IPython.display import IFrame

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
##JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "KellyReinersman"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    # Header with Logo and Identifier
    html.Div([
        html.Center([
            html.A( 
                html.Img(
                    src='data:image/png;base64,{}'.format(encoded_image.decode()),
                    style={'height': '100px', 'margin': '10px'}
                ),
                href='https://www.snhu.edu',
                target='_blank', # Opens in new tab
            ),
            html.B(html.H1('Grazioso Salvare Animal Shelter Dashboard')),
            html.H4('Kelly Reinersman - CS-340 - SNHU'),
        ])
    ]),
    html.Hr(),
    
    # Filter Options
    html.Div([
        html.Label('Select Rescue Type:', style={'font-weight': 'bold', 'font-size': '18px'}),
        html.Br(),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' Water Rescue', 'value': 'water'},
                {'label': ' Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': ' Disaster/Individual Tracking', 'value': 'disaster'},
                {'label': ' Reset (Show All)', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px', 'font-size': '16px'},
            inputStyle={'margin-right': '5px'}
        )
    ], style={'padding': '20px', 'backgroundColor': '#f0f0f0', 'borderRadius': '10px', 'margin': '10px'}),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    # Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        # Interactive features
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'padding': '10px',
            'minWidth': '100px',
            'maxWidth': '200px',
            'whiteSpace': 'normal'
        },
        style_header={
            'backgroundColor': '#2c3e50',
            'color': 'white',
            'fontWeight': 'bold'
        },
        style_data_conditional=[
            {
                'if': {'row_index': 'odd'},
                'backgroundColor': '#f9f9f9'
            }
        ]
    ),
    
    html.Br(),
    html.Hr(),
    
    # Chart and Map side by side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%', 'padding': '5px'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%', 'padding': '5px'}
            )
        ]
    )
])
#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)

# Display the breeds of animal based on quantity represented in
# the data table
def update_dashboard(filter_type):
    """Filter the data..."""
    
    if filter_type == 'water':
        # Water Rescue criteria
        df_filtered = pd.DataFrame.from_records(
            db.read({
                "animal_type": "Dog",
                "breed": {"$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]},
                "sex_upon_outcome": "Intact Female",
                "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
            })
        )
        
    elif filter_type == 'mountain':
        # Mountain/Wilderness Rescue criteria
        df_filtered = pd.DataFrame.from_records(
            db.read({
                "animal_type": "Dog",
                "breed": {"$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]},
                "sex_upon_outcome": "Intact Male",
                "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
            })
        )
        
    elif filter_type == 'disaster':
        # Disaster/Individual Tracking criteria
        df_filtered = pd.DataFrame.from_records(
            db.read({
                "animal_type": "Dog",
                "breed": {"$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]},
                "sex_upon_outcome": "Intact Male",
                "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
            })
        )
        
    else:  # reset - show all
        df_filtered = pd.DataFrame.from_records(db.read({}))
    
    # Remove _id column if present
    if '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)
    
    df_filtered.drop_duplicates(subset=['animal_id'], inplace=True)

    return df_filtered.to_dict('records')

@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    """Display pie chart of breeds in the current view"""
    
    if viewData is None or len(viewData) == 0:
        return [html.P("No data available for chart")]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Group by breed and count
    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    
    # Only show top 10 breeds, group rest as "Other"
    if len(breed_counts) > 10:
        top_breeds = breed_counts.head(10)
        other_count = breed_counts.tail(len(breed_counts) - 10)['count'].sum()
        other_row = pd.DataFrame({'breed': ['Other'], 'count': [other_count]})
        breed_counts = pd.concat([top_breeds, other_row], ignore_index=True)
    
    # Create pie chart with better formatting
    return [
        dcc.Graph(
            figure=px.pie(
                breed_counts,
                values='count',
                names='breed',
                title='<b>Breed Distribution</b>',
                hole=0.3,  # Donut style - easier to read
                color_discrete_sequence=px.colors.qualitative.Set3  # Nice color palette
            ).update_traces(
                textposition='inside',
                textinfo='percent+label',
                textfont_size=12,
                marker=dict(line=dict(color='white', width=2))  # White borders between slices
            ).update_layout(
                showlegend=True,
                legend=dict(
                    orientation='v',
                    yanchor='middle',
                    y=0.5,
                    xanchor='left',
                    x=1.01,
                    font=dict(size=10)
                ),
                title_x=0.5,  # Center title
                title_font_size=18,
                margin=dict(t=50, b=20, l=20, r=120),  # More room for legend
                height=450
            )
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '500px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
print("Layout created successfully!")
print(f"DataFrame has {len(df)} records")
app.run_server(
    mode='external',
    host='0.0.0.0',
    port=8075,
    height=1000,
    width='100%'
)

Layout created successfully!
DataFrame has 59977 records
Dash app running on http://0.0.0.0:8075/


In [2]:
from IPython.display import IFrame

IFrame('https://polygonportal-agathaguest-8075.codio.io', width='100%', height=900)